Import dependencies

In [37]:
import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams,Distance,SparseVectorParams,Modifier,PayloadSchemaType,PointStruct,Document,Prefetch,FusionQuery

Create Qdrant Collections for hybrid search

In [38]:
qdrant_client= QdrantClient(url="http://localhost:6333") 

In [39]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
         "text-embedding-3-small":VectorParams(size=1536,distance=Distance.COSINE)
        },
        sparse_vectors_config={
            "bm25":SparseVectorParams(modifier=Modifier.IDF)
        }


)

True

In [40]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

Embedding Functions

In [41]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [42]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

Read the sampled dataset with  Amazon inventory data

In [43]:
df_items=pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [44]:
df_items.head(5)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,3 Pack Fitbit Luxe Bands Compatible with Fitbi...,4.4,261,"[Buckle closure, 【Applicable Model】Meliya repl...",[Meliya Fitbit Luxe wristbands make you more f...,8.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Meliya,"[Electronics, Wearable Technology, Arm & Wrist...","{'Date First Available': 'September 19, 2022',...",B0BFQH6535,NaN,NaN,NaN
1,All Electronics,Charger Charging Cable Cord fits for Beats by ...,4.2,180,[[Perfect compatibility]: Compatible with Beat...,[],11.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],FlickerStar,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '4.33 x 2.05 x 1.54 inc...,B09TBF91FL,NaN,NaN,NaN
2,All Electronics,"Wireless Keyboard and Mouse Combo, Giecy 2.4G ...",4.4,135,[【Comfortable Wireless Keyboard Mouse Combo】 w...,[],36.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Comfortable Wireless Keyboard Mous...,Giecy,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '16.9 x 4.9 x 2.5 inche...,B0C36WD52F,NaN,NaN,NaN
3,Cell Phones & Accessories,HoneyAKE Case for Google Pixel Buds A-Series C...,3.9,126,[For Google Pixel Buds A-Series 2021 Case for ...,[],10.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Solid and not bully', 'url': 'http...",HoneyAKE,"[Electronics, Headphones, Earbuds & Accessorie...","{'Brand': 'HoneyAKE', 'Item model number': 'GU...",B09XH2TK3R,NaN,NaN,NaN
4,Computers,"2 Pack LCD Writing Tablet 12 Inch for Kids,Toy...",4.4,752,[Color drawing boards: We firmly believe that ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Newest 2 Pack LCD writing tablet f...,LEYAOYAO,"[Electronics, Computers & Accessories, Compute...","{'Standing screen display size': '12 Inches', ...",B0BFR7JRQS,NaN,NaN,NaN


Preprocess Titles and features

In [45]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"


In [46]:
def first_largeimage(row):
    return row['images'][0].get("large","")

In [47]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(first_largeimage, axis=1)


In [48]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/411HNQcSv+L._AC_.jpg')

Sample 50 items from the dataset

In [49]:
df_sample = df_items.sample(n=50, random_state=42)

In [50]:
df_data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price", "average_rating","parent_asin"]]

In [51]:
df_data_to_embed.head(5)


,preprocessed_description,image,rating_number,price,average_rating,parent_asin
0,3 Pack Fitbit Luxe Bands Compatible with Fitbi...,https://m.media-amazon.com/images/I/411HNQcSv+...,261,8.99,4.4,B0BFQH6535
1,Charger Charging Cable Cord fits for Beats by ...,https://m.media-amazon.com/images/I/41m9Y41SAf...,180,11.98,4.2,B09TBF91FL
2,"Wireless Keyboard and Mouse Combo, Giecy 2.4G ...",https://m.media-amazon.com/images/I/41tv9QvDaZ...,135,36.98,4.4,B0C36WD52F
3,HoneyAKE Case for Google Pixel Buds A-Series C...,https://m.media-amazon.com/images/I/41zgEuCGnL...,126,10.99,3.9,B09XH2TK3R
4,"2 Pack LCD Writing Tablet 12 Inch for Kids,Toy...",https://m.media-amazon.com/images/I/51Veurn-wF...,752,19.99,4.4,B0BFR7JRQS


In [52]:
data_to_embed=df_data_to_embed.to_dict(orient="records")

In [53]:
data_to_embed

[{'preprocessed_description': '3 Pack Fitbit Luxe Bands Compatible with Fitbit Luxe Women Men, Soft Silicone Wristband Replacement Strap for Fitbit Luxe/Luxe Special Edition Fitness Tracker Buckle closure 【Applicable Model】Meliya replacement wristbands Specially Designed for Fitbit Luxe / Fitbit Luxe Special Edition Only, NO Fitbit Luxe tracker included. 【Seamless Match】Fits as good as original Fitbit Luxe band. Simple to install and remove, NO tools required! The end of the band has a lug, which fit your Fitbit Luxe smart watch in place and locked precisely & securely. 【Premium Waterproof Material】Made from flexible high-quality elastomer, sweatproof & waterproof. Prevents skin from irritation, soft, lightweight and durable, very comfortable to wear. 【Available in Two Sizes】Small size: for 5.5"-7.3" wrists. Large size: for 6.7"-8.3" wrists 【After-sales Service】If you have any dissatisfaction with replacement bands for Fitbit Luxe from Meliya Store, please contact us via Amazon e-mail.

In [54]:
len(data_to_embed)

1000

In [55]:
text_to_embed=[item["preprocessed_description"] for item in data_to_embed]

In [56]:
text_to_embed

['3 Pack Fitbit Luxe Bands Compatible with Fitbit Luxe Women Men, Soft Silicone Wristband Replacement Strap for Fitbit Luxe/Luxe Special Edition Fitness Tracker Buckle closure 【Applicable Model】Meliya replacement wristbands Specially Designed for Fitbit Luxe / Fitbit Luxe Special Edition Only, NO Fitbit Luxe tracker included. 【Seamless Match】Fits as good as original Fitbit Luxe band. Simple to install and remove, NO tools required! The end of the band has a lug, which fit your Fitbit Luxe smart watch in place and locked precisely & securely. 【Premium Waterproof Material】Made from flexible high-quality elastomer, sweatproof & waterproof. Prevents skin from irritation, soft, lightweight and durable, very comfortable to wear. 【Available in Two Sizes】Small size: for 5.5"-7.3" wrists. Large size: for 6.7"-8.3" wrists 【After-sales Service】If you have any dissatisfaction with replacement bands for Fitbit Luxe from Meliya Store, please contact us via Amazon e-mail. We will reply to you ASAP.',

In [57]:
embeddings=get_embeddings_batch(text_to_embed)


Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [58]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["preprocessed_description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [26]:
pointstructs[0].vector

{'text-embedding-3-small': [0.01142120361328125,
  0.00748443603515625,
  -0.007244110107421875,
  -0.035888671875,
  -0.031890869140625,
  -0.042999267578125,
  -0.0450439453125,
  0.0243072509765625,
  0.0271453857421875,
  -0.06817626953125,
  -0.0036487579345703125,
  -0.0159149169921875,
  -0.0297393798828125,
  -0.026947021484375,
  0.032318115234375,
  0.039337158203125,
  -0.036529541015625,
  -0.005077362060546875,
  -0.051666259765625,
  0.05487060546875,
  0.001708984375,
  0.045379638671875,
  0.003276824951171875,
  -0.049774169921875,
  0.0026340484619140625,
  -0.019805908203125,
  -0.04425048828125,
  -0.00902557373046875,
  -0.005672454833984375,
  0.059722900390625,
  0.017669677734375,
  -0.01080322265625,
  0.01038360595703125,
  -0.0325927734375,
  -0.02703857421875,
  0.033172607421875,
  0.01947021484375,
  -0.01230621337890625,
  0.03875732421875,
  0.0214691162109375,
  -0.006061553955078125,
  0.050445556640625,
  0.00634002685546875,
  0.00482940673828125,
  

In [59]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [60]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

Hybrid Retrieval

In [61]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [62]:
results = retrieve_data("Can I get a tablet?", k=20)

In [63]:
results

{'retrieved_context_ids': ['B0BN7JTB5N',
  'B0B7QL4FMN',
  'B0C4KCQB19',
  'B09MKYBJT6',
  'B0BJPTYFK9',
  'B07FMF2L6B',
  'B0BCFYCXRH',
  'B0B3QGL9C9',
  'B0BDD6BC6Y',
  'B0BQ1ZVF2C',
  'B0C6QXGW5S',
  'B0BCDYMF5L',
  'B0C1TWNBGD',
  'B0BHP1HMK2',
  'B09MFHFHQM',
  'B0B9JSJ2RF',
  'B0BKZN3N6L',
  'B0B1DFWMLK',
  'B0BGLJC82D',
  'B0B7MR7WS7'],
 'retrieved_context': ['UGREEN 2 Pack Tablet Stand Holder Adjustable Desktop Portable Stand Home Office Desk Accessories Compatible with iPad 10.2 iPad Pro 11 Inch iPad 9.7 iPad Mini 6 5 4 3 Phone White Wide Compatibility: This tablet stand for desk can fit for 4 to 11 inch tablet, cell phone and E-reader with Max thickness of 0.55" (including its case). Compatible for Apple iPad 9.7 inch, iPad 10.2" 2019, iPad Pro 11-inch 2020, iPad mini 2 3 4 5 Wi-Fi, iPad Air 2, iPhone 13 12 11 Pro Max, iPhone 12 mini, iPhone and android phones. Lightweight & Minimalist Design: With a small size of 4.7" x 4.2", the tablet stand can be foldable and easy to slip

Hybrid Search with weighted RRF

In [65]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [66]:
results = retrieve_data("Can I get a tablet?", k=20)

In [67]:
results

{'retrieved_context_ids': ['B0B7QL4FMN',
  'B0C4KCQB19',
  'B09MKYBJT6',
  'B07FMF2L6B',
  'B0BN7JTB5N',
  'B0B3QGL9C9',
  'B0BCFYCXRH',
  'B0BCDYMF5L',
  'B0C1TWNBGD',
  'B0BQ1ZVF2C',
  'B0BHP1HMK2',
  'B0BJPTYFK9',
  'B0BKZN3N6L',
  'B0B7MR7WS7',
  'B0C1P9YW5R',
  'B0BDD6BC6Y',
  'B0BTPG4XHH',
  'B09RN3KN5C',
  'B0C6QXGW5S',
  'B09SFX1KYC'],
 'retrieved_context': ['GrandPad Senior Tablet (Renewed) with Phone Capabilities, 4G LTE, Wireless Charger, Stylus, 1 Month Premium Service Plan Included, Purchase A Plan at Activation MADE FOR SENIORS: The GrandPad is the ideal tablet & phone, all-in-one connectivity device for older adults. It has a closed network, so it keeps the older adult free of receiving any spam or robocalls. Its simple interface & easy connectivity makes it the perfect choice for families looking to stay connected in a safe & secure way. SERVICE PLAN NEEDED: 1 Month Included - A Monthly or Annual Service Plan can be purchased at Activation by calling GrandPad directly. 